# Modelado de datos

In [1]:
%load_ext watermark
%watermark

Last updated: 2022-10-12T00:15:22.492427-05:00

Python implementation: CPython
Python version       : 3.7.4
IPython version      : 7.8.0

Compiler    : MSC v.1915 64 bit (AMD64)
OS          : Windows
Release     : 10
Machine     : AMD64
Processor   : Intel64 Family 6 Model 158 Stepping 9, GenuineIntel
CPU cores   : 8
Architecture: 64bit



In [2]:
import pandas as pd
%matplotlib inline 
# no crea widget ni nada para no presentar resultados.
import pandas as pd
import numpy as np
#import matplotlib.pyplot as plt
import matplotlib as plt
import seaborn as sns
from matplotlib.pyplot import figure
import warnings
warnings.filterwarnings("ignore")


plt.rcParams['figure.figsize'] = (10,10) # 10 pulgadas de ancho y 10 de alto para todos los plots

In D:\ProgramData\Anaconda3\envs\EnvDataLab\lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test.mplstyle: 
The text.latex.preview rcparam was deprecated in Matplotlib 3.3 and will be removed two minor releases later.
In D:\ProgramData\Anaconda3\envs\EnvDataLab\lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test.mplstyle: 
The mathtext.fallback_to_cm rcparam was deprecated in Matplotlib 3.3 and will be removed two minor releases later.
In D:\ProgramData\Anaconda3\envs\EnvDataLab\lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test.mplstyle: Support for setting the 'mathtext.fallback_to_cm' rcParam is deprecated since 3.3 and will be removed two minor releases later; use 'mathtext.fallback : 'cm' instead.
In D:\ProgramData\Anaconda3\envs\EnvDataLab\lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test.mplstyle: 
The validate_bool_maybe_none function was deprecated in Matplotlib 3.3 and will be removed two minor releases later.
In D:\ProgramData\Anaconda3\

In [10]:
bank_dp = pd.read_parquet("../output/data-bank_4_PreModeling.parquet",engine='pyarrow')

In [11]:
bank_dp.head(5)

,edad,flag_mora,prom_saldoAnual,flag_hipotecario,flag_prestamo,tipo_contacto,dia_UltContacto,seg_UltContacto,cant_contactoCampanha,cant_contactoBeforeCampanha,...,mes_UltContacto_nov,mes_UltContacto_oct,mes_UltContacto_sep,estadoCivil_divorced,estadoCivil_married,estadoCivil_single,educacion_primary,educacion_secondary,educacion_tertiary,educacion_unknown
0,59,1,2343,1,0,2,5,1042,1,0,...,0,0,0,0,1,0,0,1,0,0
1,56,1,45,0,0,2,5,1467,1,0,...,0,0,0,0,1,0,0,1,0,0
2,41,1,1270,1,0,2,5,1389,1,0,...,0,0,0,0,1,0,0,1,0,0
3,55,1,2476,1,0,2,5,579,1,0,...,0,0,0,0,1,0,0,1,0,0
4,54,1,184,0,0,2,5,673,2,0,...,0,0,0,0,1,0,0,0,1,0


In [12]:
'''Check numerical data set'''
display(bank_dp.head(3), bank_dp.shape)
display('Observamos 11152 registros y 73 columnas numéricas luego de la transformación.')

,edad,flag_mora,prom_saldoAnual,flag_hipotecario,flag_prestamo,tipo_contacto,dia_UltContacto,seg_UltContacto,cant_contactoCampanha,cant_contactoBeforeCampanha,...,mes_UltContacto_nov,mes_UltContacto_oct,mes_UltContacto_sep,estadoCivil_divorced,estadoCivil_married,estadoCivil_single,educacion_primary,educacion_secondary,educacion_tertiary,educacion_unknown
0,59,1,2343,1,0,2,5,1042,1,0,...,0,0,0,0,1,0,0,1,0,0
1,56,1,45,0,0,2,5,1467,1,0,...,0,0,0,0,1,0,0,1,0,0
2,41,1,1270,1,0,2,5,1389,1,0,...,0,0,0,0,1,0,0,1,0,0


(11152, 45)

'Observamos 11152 registros y 73 columnas numéricas luego de la transformación.'

## 1. Selecionar el resultado de la métrica

Se propone utilizar métricas ROC_AUC para evaluar diferentes modelos con monitoreo adicional de la dinámica de la métrica de precisión. 

Este enfoque nos permitirá explorar modelos desde diferentes ángulos.

## 2. Separar data de entrenamiento y test

In [7]:
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
import gc 

In [8]:
train_test_split?

### Muestreo Estratificado

Es un concepto importante que a menudo se pasa por alto al desarrollar un modelo, ya sea para regresión o clasificación. Recuerde, que para evitar el sobreajuste de nuestros datos, debemos implementar una validación cruzada, sin embargo, debemos asegurarnos de que al menos las características que tienen la mayor influencia en nuestra etiqueta (si un cliente potencial abrirá un depósito a plazo o no) es igualmente distribuido. ¿Qué quiero decir con esto?

In [16]:
bank_dp.result_campanha.unique()

array([0, 1], dtype=uint8)

In [13]:
bank_dp.columns

Index(['edad', 'flag_mora', 'prom_saldoAnual', 'flag_hipotecario',
       'flag_prestamo', 'tipo_contacto', 'dia_UltContacto', 'seg_UltContacto',
       'cant_contactoCampanha', 'cant_contactoBeforeCampanha',
       'result_campanha', 'FlagTarget_DP', 'Rango_edad', 'duration_status',
       'trabajo_admin.', 'trabajo_blue-collar', 'trabajo_entrepreneur',
       'trabajo_housemaid', 'trabajo_management', 'trabajo_retired',
       'trabajo_self-employed', 'trabajo_services', 'trabajo_student',
       'trabajo_technician', 'trabajo_unemployed', 'trabajo_unknown',
       'mes_UltContacto_apr', 'mes_UltContacto_aug', 'mes_UltContacto_dec',
       'mes_UltContacto_feb', 'mes_UltContacto_jan', 'mes_UltContacto_jul',
       'mes_UltContacto_jun', 'mes_UltContacto_mar', 'mes_UltContacto_may',
       'mes_UltContacto_nov', 'mes_UltContacto_oct', 'mes_UltContacto_sep',
       'estadoCivil_divorced', 'estadoCivil_married', 'estadoCivil_single',
       'educacion_primary', 'educacion_secondary', 'e

Por ejemplo para el préstamo personal `flag_prestamo`:

Tener un préstamo personal es una característica importante que determina si un cliente potencial abrirá un depósito a plazo o no

Pasos para implementar muestreo estratificado:

- 1) Necesitamos ver cómo se distribuyen nuestros datos. 
- 2) Después de notar que la columna de préstamo contiene 87% de "no" (No tiene préstamos personales) y 13% de "sí" (Tiene préstamos personales). 
- 3) Queremos asegurarnos de que nuestro conjunto de entrenamiento y prueba contenga la misma proporción de 87% "no" y 13% "sí".

Distribuciión de `flag_prestamo`

In [14]:
bank_dp["flag_prestamo"].value_counts()/len(bank_dp)

0    0.86944
1    0.13056
Name: flag_prestamo, dtype: float64

In [15]:
X = bank_dp.drop(columns = 'FlagTarget_DP')
y = bank_dp[['FlagTarget_DP']]

In [29]:
gc?

In [18]:
''' Split data on train and test'''
# set global random state
random_state = 42
# split data
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state ,shuffle=False)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=random_state)
# collect excess data
gc.collect()

11

In [32]:
train_test_split?

Comprobar el muestreo estratificado

In [20]:
X_train["flag_prestamo"].value_counts()/len(X_train)

0    0.870194
1    0.129806
Name: flag_prestamo, dtype: float64

In [21]:
y_train["FlagTarget_DP"].value_counts()/len(y_train)

0    0.525838
1    0.474162
Name: FlagTarget_DP, dtype: float64

In [48]:
X_test["flag_prestamo"].value_counts()/len(X_test)

0    0.866428
1    0.133572
Name: flag_prestamo, dtype: float64

In [22]:
y_test["FlagTarget_DP"].value_counts()/len(y_test)

0    0.526221
1    0.473779
Name: FlagTarget_DP, dtype: float64

Si se compara la distribución del dataset para (x,y) train como para test las distribuciones son similares, de esta manera vemos que los datos una distribución similarmente alta.

In [23]:
display('El tamaño de los datos sepadaso en train y test son:', X_train.shape, y_train.shape, X_test.shape, y_test.shape)

'El tamaño de los datos sepadaso en train y test son:'

(8921, 44)

(8921, 1)

(2231, 44)

(2231, 1)

## 3. Modelado

### a. Árbol de decisión C 5.0